<a href="https://colab.research.google.com/github/EunjeLee0812/Sanhak/blob/seowonryeol/code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 라이브러리 설치
!pip install faster-whisper rapidfuzz g2pk konlpy python-mecab-ko

In [ ]:
import gc, sys
import os, re, json, glob, csv, random, glob, time
from dataclasses import dataclass
from typing import Dict, List, Optional, Any, Tuple
from g2pk import G2p
from faster_whisper import WhisperModel
from rapidfuzz.distance import Levenshtein
from rapidfuzz import process, fuzz
from mecab import MeCab
import importlib

In [ ]:
# #Googledrive 마운트(Colab 사이트 사용 시 주석 해제)
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# 1. 파일들이 위치한 경로로 이동 colab용 
# BASE_PATH = "/content/drive/MyDrive/25-2 산학협력프로젝트/26.1_최종발표/results/"
BASE_PATH = "/teamspace/studios/this_studio/"
%cd {BASE_PATH}

# 모듈 Import
from config.settings import *
from utils.normalizer import TextNormalizer
from utils.data_loader import load_transcripts
from utils.metrics import calculate_cer, calculate_wer, evaluate_proper_nouns
from core.asr_engine import ASR
from core.bias_manager import BiasManager
from core.post_processor import postprocess_with_hotwords


In [ ]:
# #그래픽카드 메모리 남용을 막기 위한 캐시 초기화

# gc.collect()
# torch.cuda.empty_cache()

# 1-5. 결과 저장 경로[현재 시간 반영해서 파일별 구분 용이]
#results 폴더 없으면 생성
if not os.path.exists(os.path.join(BASE_PATH,"results")):
    os.makedirs(os.path.join(BASE_PATH,"results"))

now = time.gmtime(time.time()+(9*3600)) #한국 시간
formatted = time.strftime("[%Y%m%d_%H%M]", now)
OUT_ROWS = f"./results/{formatted}_asr_detail.csv"
OUT_SUM  = f"./results/{formatted}_asr_summary.csv"

def summarize(rows: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    # key는 4개이므로 Tuple[int, int, str, int]
    agg: Dict[Tuple[int, int, str, int], Dict[str, Any]] = {}

    for r in rows:
        hotwords_strategy = r.get("hotwords_strategy", "random")
        bias_weight_update_cnt = int(r.get("bias_weight_update_cnt", 0))

        key = (
            int(r["top_k"]),
            int(r["postprocess_on"]),   # 🔥 postprocess_on → postprocess_on
            hotwords_strategy,
            bias_weight_update_cnt,
        )

        a = agg.setdefault(
            key,
            {
                "top_k": key[0],
                "postprocess_on": key[1],
                "hotwords_strategy": key[2],
                "bias_weight_update_cnt": key[3],
                "files_num": 0,
                "cer_sum": 0.0,
                "wer_sum": 0.0,
                "pn_recall_sum": 0.0,
                "pn_cer_sum": 0.0,
                "wrong_char_sum":0,
                "wrong_word_sum":0, 
                "char_sum":0, 
                "word_sum":0
            },
        )

        #파일 개수 및 전체 cer, wer 계산
        a["files_num"] += 1
        a["wrong_char_sum"] += float(r["wrong_char_cnt"])
        a["wrong_word_sum"] += float(r["wrong_word_cnt"])
        a["char_sum"]+=r["char_cnt"]
        a["word_sum"]+=r["word_cnt"]

        if r.get("pn_recall") is not None:
            a["pn_recall_sum"] += float(r["pn_recall"])
            a["pn_cer_sum"] += float(r["pn_cer"])

    out = []
    for _, a in sorted(agg.items()):
        n = max(1, a["files_num"])
        out.append({
                "top_k": a["top_k"],
                "postprocess_on": a["postprocess_on"],
                "hotwords_strategy": a["hotwords_strategy"],
                "bias_weight_update_cnt": a["bias_weight_update_cnt"],
                "used_file_num": AUDIO_FILE_MAX,
                # round(값, 4)를 통해 소수점 4자리까지 반올림합니다.
                "cer": round(a["wrong_char_sum"]/max(a["char_sum"],1), 4),
                "wer": round(a["wrong_word_sum"] / max(1, a["word_sum"]), 4),
                "pn_recall_avg": round(a["pn_recall_sum"] / max(1, a["files_num"]), 4),
                "pn_cer_avg": round(a["pn_cer_sum"] / max(1, a["files_num"]), 4)
            })

    return out

# ==============================================================================
# 메인 실행 로직
# ==============================================================================

# 1. 초기화 및 로드
if not os.path.exists(BASE_DIR):
    print("[WARN] Base path not found. Checking local..")

normalizer = TextNormalizer()
mecab = MeCab()
bias_mgr = BiasManager(BIAS_PATH)
transcripts = load_transcripts(TRANSCRIPTS_PATH)
files = glob.glob(os.path.join(AUDIO_FOLDER, "**/*.mp4"), recursive=True)[:AUDIO_FILE_MAX]

# ASR 모델 로드
asr = ASR(ASR_MODEL, ASR_DEVICE, ASR_COMPUTE,initial_prompt=KOREAN_ONLY_PROMPT)

rows: List[Dict[str, Any]] = []  # [수정] 결과 데이터를 저장할 리스트

# 2. 실험 루프
for top_k in HOTWORD_TOPK_SWEEP: #hotwords 개수 경우의 수 반복문
    for hotwords_strategy in HOTWORD_STRATEGY_SWEEP: #hotwords 선택 전략 경우의 수 반복문

        # (선택) 각 전략 시작마다 bias 초기화
        if RESET_BIASING_LIST:
            bias_mgr.reset_biasing_list(BIAS_PATH)
        for bias_weight_update_cnt in BIAS_WEIGHT_UPDATE_CYCLE_SWEEP: #bias_weight_update 주기 경우의 수 반복문
            # ✅ 1번 방식: 반복 횟수는 BIAS_WEIGHT_UPDATE_CYCLE_SWEEP
            for repeat in range(bias_weight_update_cnt):

                # ✅ repeat마다 hotwords 새로 샘플링 (1번 방식)
                current_hotwords = bias_mgr.get_weighted_hotwords(top_k, mode=hotwords_strategy)
                pp_str= "ON" if pp_on ==1 else "OFF"
                for pp_on in POSTPROCESS_SWEEP:
                    print(f"\n[RUN] Top-K: {top_k} | Iteration: {repeat+1}/{bias_weight_update_cnt} | PostProcess: {pp_str}")
                    print(f"hotwords : {current_hotwords}\n")

                    for audio_path in files:
                        fname = os.path.basename(audio_path)
                        meta = transcripts.get(fname, {"text": "", "entities": []})

                        # 1) ASR
                        hyp_raw = asr.transcribe(audio_path, "ko", ASR_BEAM, hotwords=current_hotwords)

                        # 2) 후처리
                        if pp_on:
                            hyp_final, replog = postprocess_with_hotwords(
                                hyp_raw, current_hotwords, normalizer,
                                gate=RULE_GATE, tol=RULE_TOL, wratio_th=RULE_WRATIO_TH
                            )
                        else:
                            hyp_final, replog = hyp_raw, []

                        # 3) Metrics
                        cer, wrong_char_cnt, char_cnt = calculate_cer(meta["text"], hyp_final, normalizer)
                        wer, wrong_word_cnt, word_cnt = calculate_wer(meta["text"], hyp_final, normalizer, mecab)
                        # ✅ 4) PN 평가: 너가 수정한 4개 리턴 버전 사용
                        pn_recall, pn_cer, hyp_ents, hard_missed_ents = evaluate_proper_nouns(
                            meta.get("entities", []), hyp_final, normalizer, match_th=PN_MATCH_TH, hard_th=HARD_MISS_TH)

                        # ✅ 1번 방식: hard miss만 학습
                        bias_mgr.add_miss(hard_missed_ents)

                        pn_recall = pn_recall if pn_recall is not None else 0.0

                        # 로그
                        print(
                            f"- file: {os.path.dirname(audio_path).split('/')[-1]}/{fname} | "
                            f"pp_on={pp_on} | cer={cer:.4f} | wer={wer:.4f} | pn_cer={pn_cer:.4f} | pn_recall={pn_recall:.4f}"
                        )
                        print(
                            f"ref_text:  [{meta['text']}]\n"
                            f"hyp_raw:   [{normalizer.normalize(hyp_raw)}]\n"
                            f"hyp_final: [{hyp_final}]\n"
                            f"ref_pn:    {meta.get('entities', [])}\n"
                            f"hyp_pn:    {hyp_ents}\n"
                            f"hard_miss: {hard_missed_ents}\n"
                        )

                        # 결과 저장(컬럼명 정리 권장)
                        rows.append({
                            "file": f"{os.path.dirname(audio_path).split('/')[-1]}/{fname}",
                            "top_k": top_k,
                            "postprocess_on": int(pp_on),  # 이름 명확히
                            "hotwords_strategy": "random" if hotwords_strategy == 1 else "hybrid",
                            "hotwords": current_hotwords,
                            "bias_weight_update_cnt": repeat+1,  # 

                            "cer": f"{cer:.4f}",
                            "wer": f"{wer:.4f}",
                            "pn_recall": f"{pn_recall:.4f}",
                            "pn_cer": f"{pn_cer:.4f}",

                            "ref_text": meta["text"],
                            "hyp_raw": normalizer.normalize(hyp_raw),
                            "hyp_final": hyp_final,

                            "ref_text_pn": meta.get("entities", []),
                            "hyp_pn": hyp_ents,
                            "hard_missed_pn": hard_missed_ents,

                            "replog": json.dumps(replog, ensure_ascii=False),
                            "wrong_char_cnt":wrong_char_cnt,
                            "char_cnt":char_cnt,
                            "wrong_word_cnt":wrong_word_cnt,
                            "word_cnt":word_cnt
                        })

        # repeat 끝에 학습 반영
        bias_mgr.finalize()

with open(OUT_ROWS, "w", newline="", encoding="utf-8-sig") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    w.writeheader()
    w.writerows(rows)

summary = summarize(rows)
with open(OUT_SUM, "w", newline="", encoding="utf-8-sig") as f:
    w = csv.DictWriter(f, fieldnames=list(summary[0].keys()))
    w.writeheader()
    w.writerows(summary)

print("\n[DONE] All experiments finished.")